# KADMON Optuna — Binning + Sinkhorn

Le classement utilise `global_distance_mm`. L'objectif minimise le rapport entre distance intra-identité et distance moyenne inter-identité.

- Compression : binning CPU mis en cache
- Transport : Sinkhorn débiaisé sur GPU
- Étude : `binning_sinkhorn`

## 1. Imports et configuration


In [1]:
import gc
import hashlib
import json
import os
from pathlib import Path
import sys
from time import perf_counter
import warnings
import numpy as np
import optuna
import pandas as pd
import torch
from IPython.display import display
from optuna.trial import TrialState
from plotly.io import show

WORKING_DIR = Path.cwd().resolve()
KADMON_ROOT = next(
    (path for path in (WORKING_DIR, *WORKING_DIR.parents)
     if (path / 'kadmon').is_dir() and (path / 'notebooks').is_dir()),
    None,
)
if KADMON_ROOT is None:
    raise RuntimeError(f'Racine KADMON introuvable depuis {WORKING_DIR}.')
NOTEBOOK_DIR = KADMON_ROOT / 'notebooks' / 'optuna'
BUNDLES_DIR = KADMON_ROOT / 'notebooks' / 'bundles'
STUDY_PATH = NOTEBOOK_DIR / 'studies' / 'optuna_reid.sqlite3'
COMPRESSION_CACHE_DIR = NOTEBOOK_DIR / 'cache' / 'binning_cpu'
if str(KADMON_ROOT) not in sys.path: sys.path.insert(0, str(KADMON_ROOT))
from kadmon.compression import compress_binning
from kadmon.io import BundleCollection
from kadmon.protocol import HCP_REID_PROTOCOL
from kadmon.reid import aggregate_reid_metrics, bundle_reid_metrics, set_trial_metrics
from kadmon.selection import select_best_reid_trial
from kadmon.optimization import (
    StudyRunPolicy, create_reid_study, fail_stale_running_trials,
    run_until_complete,
)
from kadmon.cpu import evaluate_pair_cpu as evaluate_sinkhorn_pair_cpu
from kadmon.gpu import (
    evaluate_pair_gpu as evaluate_sinkhorn_pair_gpu,
    release_gpu_memory,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)
EXPERIMENT_NAME, COMPRESSION, TRANSPORT, N_TRIALS = 'binning_sinkhorn', 'binning', 'sinkhorn', 120
# Réglages conservateurs: ce notebook privilégie la stabilité aux pics de débit.
GPU_DTYPE, GPU_MDF_BATCH_SIZE = torch.float32, 32
GPU_MEMORY_FRACTION = 0.55
# OT est dense/quadratique. Au-delà de cette limite, un essai est élagué
# avant de construire les matrices (6 000² float32 = 137 Mio par matrice).
MAX_REPRESENTATIVES = 6_000
GPU_DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
GPU_BACKEND_VALIDATED = False
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(GPU_DEVICE)
    print(f'[GPU] backend=PyTorch/POT, device={torch.cuda.get_device_name(GPU_DEVICE)}')
    print(f'[GPU] mémoire={free/2**30:.2f}/{total/2**30:.2f} Gio, dtype={GPU_DTYPE}')
    print('[GPU] MDF/coût=torch_mdf_pairwise_batched; OT=ot.sinkhorn backend torch')
else:
    warnings.warn('CUDA indisponible : fallback CPU explicite.', RuntimeWarning)


Info: some functions in tractosearch.resampling are faster when 'numba' is installed


/tmp/ipykernel_89802/32140757.py:61: RuntimeWarning: CUDA indisponible : fallback CPU explicite.
  warnings.warn('CUDA indisponible : fallback CPU explicite.', RuntimeWarning)


## 2. Données et compression Binning CPU mise en cache

In [2]:
REFERENCE_SUBJECT = HCP_REID_PROTOCOL.reference_subject
INTRA_IDENTITY_SUBJECT = HCP_REID_PROTOCOL.intra_identity_subject
COMPARISON_SUBJECTS = HCP_REID_PROTOCOL.comparison_subjects
SUBJECTS = HCP_REID_PROTOCOL.subjects
N_POINTS, SEED, BUNDLES_TO_RUN = HCP_REID_PROTOCOL.n_points, 42, None
# Un seul worker évite que loky recopie plusieurs bundles volumineux en RAM.
N_JOBS_CPU, CACHE_VERSION = 1, 'binning-oriented-v2'

bundles = BundleCollection(
    BUNDLES_DIR, SUBJECTS, n_points=N_POINTS, selected=BUNDLES_TO_RUN
)
SUBJECT_FILES = bundles.files
bundle_names = bundles.names
# Les .npy restent memory-mappés; seul le bundle en cours est gardé en mémoire.
bundle_cache = bundles.cache
load_bundle = bundles.load

def effective_parameters(subject, bundle_name, parameters):
    return dict(parameters)

def compression_key(subject, bundle_name, parameters):
    path, stat = SUBJECT_FILES[subject][bundle_name], SUBJECT_FILES[subject][bundle_name].stat()
    metadata = {'version':CACHE_VERSION,'subject':subject,'bundle':bundle_name,'path':str(path.resolve()),'size':stat.st_size,'mtime_ns':stat.st_mtime_ns,'parameters':sorted(parameters.items()),'n_points':N_POINTS}
    digest = hashlib.sha256(json.dumps(metadata,sort_keys=True).encode()).hexdigest()[:24]
    return digest, metadata

def compress_one_cpu(subject, bundle_name, parameters):
    digest, metadata = compression_key(subject,bundle_name,parameters)
    path = COMPRESSION_CACHE_DIR / f'{digest}.npz'
    if path.exists():
        with np.load(path,allow_pickle=False) as z:
            if str(z['metadata']) == json.dumps(metadata,sort_keys=True):
                return (subject,bundle_name),(z['representatives'],z['weights']),True
    reps, weights = compress_binning(np.asarray(load_bundle(subject,bundle_name),dtype=np.float64),**parameters)
    COMPRESSION_CACHE_DIR.mkdir(parents=True,exist_ok=True)
    temporary = path.with_suffix('.tmp.npz')
    np.savez_compressed(temporary,representatives=reps,weights=weights,metadata=json.dumps(metadata,sort_keys=True))
    os.replace(temporary,path)
    return (subject,bundle_name),(reps,weights),False

def prepare_bundle_compressions_cpu(parameters, bundle_name):
    # Retourne exactement 11 distributions; aucune accumulation entre bundles.
    result, disk_hits = {}, 0
    for subject in SUBJECTS:
        effective = effective_parameters(subject,bundle_name,parameters)
        key, distribution, disk_hit = compress_one_cpu(subject,bundle_name,effective)
        n_representatives = len(distribution[0])
        if n_representatives > MAX_REPRESENTATIVES:
            del result, distribution
            bundle_cache.clear(); gc.collect()
            raise MemoryError(f'{bundle_name}/{subject}: {n_representatives} représentants; limite dense={MAX_REPRESENTATIVES}')
        result[key] = distribution
        disk_hits += int(disk_hit)
    return result, disk_hits

print(f'Données={BUNDLES_DIR}; bundles={len(bundle_names)}; workers CPU={N_JOBS_CPU}')


Données=/home/colin/Tractographie/KADMON/notebooks/bundles; bundles=31; workers CPU=1


## 3. MDF batché et Sinkhorn débiaisé GPU

In [3]:
def evaluate_pair_gpu(source_distribution, target_distribution, parameters, log=False, return_cost=False):
    return evaluate_sinkhorn_pair_gpu(
        source_distribution,
        target_distribution,
        parameters,
        device=GPU_DEVICE,
        dtype=GPU_DTYPE,
        batch_size=GPU_MDF_BATCH_SIZE,
        memory_fraction=GPU_MEMORY_FRACTION,
        return_cost=return_cost,
        log=log,
    )


def evaluate_pair_cpu(source_distribution, target_distribution, parameters):
    return evaluate_sinkhorn_pair_cpu(
        source_distribution,
        target_distribution,
        parameters,
        memory_fraction=0.50,
    )


release_trial_gpu = release_gpu_memory


## 4. Validation CPU/GPU obligatoire
Le plus petit bundle commun est testé sur les dix candidats; coûts, OT, poids et ratio objectif sont comparés.

In [4]:
VALIDATION_BINNING = {'bin_size':8.0,'binning_nb':2,'method':'median','n_points':N_POINTS}
VALIDATION_OT = {'epsilon':.75,'max_iter':2000,'stop_threshold':1e-6,'reject_threshold':1e-5}
GPU_VALIDATION_RESULTS = pd.DataFrame()

def validate_gpu_backend():
    global GPU_BACKEND_VALIDATED,GPU_VALIDATION_RESULTS
    if not torch.cuda.is_available(): return False
    pilot = min(bundle_names,key=lambda name:sum(len(load_bundle(s,name)) for s in SUBJECTS))
    distributions,_ = prepare_bundle_compressions_cpu(VALIDATION_BINNING,pilot)
    rows,cpu_dist,gpu_dist = [],[],[]
    for i,candidate in enumerate(COMPARISON_SUBJECTS):
        source = distributions[(REFERENCE_SUBJECT,pilot)]; target = distributions[(candidate,pilot)]
        cpu = evaluate_pair_cpu(source,target,VALIDATION_OT); gpu = evaluate_pair_gpu(source,target,VALIDATION_OT,log=(i==0),return_cost=True)
        difference = np.abs(cpu['cost']-gpu['cost']); cpu_dist.append(cpu['global_distance_mm']); gpu_dist.append(gpu['global_distance_mm'])
        rows.append({'bundle':pilot,'candidate':candidate,'cost_max_abs':difference.max(),'cost_mean_abs':difference.mean(),'ot_distance_abs':abs(cpu['global_distance_mm']-gpu['global_distance_mm']),'ot_objective_abs':abs(cpu['objective']-gpu['objective']),'source_weight_max_abs':np.max(np.abs(source[1]-gpu['weights'][0])),'target_weight_max_abs':np.max(np.abs(target[1]-gpu['weights'][1]))})
    intra = COMPARISON_SUBJECTS.index(INTRA_IDENTITY_SUBJECT)
    cpu_ratio = cpu_dist[intra]/np.delete(cpu_dist,intra).mean(); gpu_ratio = gpu_dist[intra]/np.delete(gpu_dist,intra).mean()
    GPU_VALIDATION_RESULTS = pd.DataFrame(rows); display(GPU_VALIDATION_RESULTS)
    print(f'[validation] coût max={GPU_VALIDATION_RESULTS.cost_max_abs.max():.3e}, moyen={GPU_VALIDATION_RESULTS.cost_mean_abs.mean():.3e}')
    print(f'[validation] OT max={GPU_VALIDATION_RESULTS.ot_distance_abs.max():.3e}; poids max={GPU_VALIDATION_RESULTS[["source_weight_max_abs","target_weight_max_abs"]].to_numpy().max():.3e}')
    print(f'[validation] ratio CPU={cpu_ratio:.8f}, GPU={gpu_ratio:.8f}, diff={abs(cpu_ratio-gpu_ratio):.3e}')
    GPU_BACKEND_VALIDATED = bool(GPU_VALIDATION_RESULTS.cost_max_abs.max()<=2e-4 and np.allclose(cpu_dist,gpu_dist,rtol=5e-4,atol=1e-5) and np.isclose(cpu_ratio,gpu_ratio,rtol=5e-4,atol=1e-5) and GPU_VALIDATION_RESULTS[['source_weight_max_abs','target_weight_max_abs']].to_numpy().max()==0)
    if not GPU_BACKEND_VALIDATED: warnings.warn('Validation CPU/GPU échouée: fallback CPU.',RuntimeWarning)
    else: print('[GPU] Validation réussie; backend GPU activé.')
    del distributions
    bundle_cache.clear()
    release_trial_gpu(); return GPU_BACKEND_VALIDATED

validate_gpu_backend()


False

## 5. Objectif Optuna séquentiel sur GPU

In [5]:
def sample_parameters(trial):
    # Les tailles 2--7 mm créent jusqu'à ~30k représentants sur ces données:
    # le Sinkhorn dense correspondant ne peut pas tenir sur un GPU de 16 Gio.
    return (
        {
            'bin_size': trial.suggest_float('bin_size', 8.0, 16.0, step=1.0),
            'binning_nb': trial.suggest_int('binning_nb', 2, 3),
            'method': trial.suggest_categorical('method', ['mean', 'median']),
            'n_points': N_POINTS,
        },
        {
            'epsilon': trial.suggest_float('epsilon', 1e-3, 1.0, log=True),
            'max_iter': 2000, 'stop_threshold': 1e-6, 'reject_threshold': 1e-5,
        },
    )

def evaluate_bundle(bundle_name, distributions, parameters, use_gpu):
    pair_metrics=[]
    for i,candidate in enumerate(COMPARISON_SUBJECTS):
        function = evaluate_pair_gpu if use_gpu else evaluate_pair_cpu
        kwargs = {'log':i==0} if use_gpu else {}
        source_distribution = distributions[(REFERENCE_SUBJECT,bundle_name)]
        target_distribution = distributions[(candidate,bundle_name)]
        value = function(source_distribution,target_distribution,parameters,**kwargs)
        pair_metrics.append({'global_distance_mm':value['global_distance_mm'],'mean_mm':value['mean_mm'],'mass':value['mass']})
        del value
        if use_gpu: torch.cuda.empty_cache()
    return bundle_reid_metrics(
        bundle_name, [row['global_distance_mm'] for row in pair_metrics],
        comparison_subjects=COMPARISON_SUBJECTS,
        intra_identity_subject=INTRA_IDENTITY_SUBJECT,
        mean_displacement_mm=np.mean([row['mean_mm'] for row in pair_metrics]),
        mean_transported_mass=np.mean([row['mass'] for row in pair_metrics]),
        mean_n_representatives=np.mean([len(distributions[(s,bundle_name)][0]) for s in SUBJECTS]),
    )

def evaluate_trial_gpu(trial,compression_parameters,transport_parameters):
    use_gpu=GPU_BACKEND_VALIDATED
    if not use_gpu:
        warnings.warn(
            '[CPU fallback] CUDA absent ou validation échouée; évaluation CPU séquentielle.',
            RuntimeWarning,
        )
    results=[]; cache_hits=0
    for bundle_index,name in enumerate(bundle_names,1):
        distributions,hits = prepare_bundle_compressions_cpu(compression_parameters,name)
        cache_hits += hits
        results.append(evaluate_bundle(name,distributions,transport_parameters,use_gpu))
        del distributions
        bundle_cache.clear()
        release_trial_gpu()
        trial.report(float(np.mean([row['intra_inter_ratio'] for row in results])),bundle_index)
        if trial.should_prune(): raise optuna.TrialPruned(f'Pruning après {bundle_index} bundles')
    print(f'[CPU] cache disque: {cache_hits}/{len(bundle_names)*len(SUBJECTS)} compression(s)')
    print(f'[GPU] Trial {trial.number} completed' if use_gpu else f'[CPU fallback] Trial {trial.number} completed')
    return pd.DataFrame(results),use_gpu

def objective(trial):
    started = perf_counter()
    compression_parameters, transport_parameters = sample_parameters(trial)
    try: bundle_metrics,use_gpu=evaluate_trial_gpu(trial,compression_parameters,transport_parameters)
    except (RuntimeError,MemoryError) as exc:
        if isinstance(exc,(MemoryError,torch.cuda.OutOfMemoryError)) or ('Sinkhorn' in str(exc) and 'convergé' in str(exc)): raise optuna.TrialPruned(str(exc)) from exc
        raise
    finally:
        bundle_cache.clear()
        release_trial_gpu()
    aggregates = aggregate_reid_metrics(
        bundle_metrics, n_comparison_subjects=len(COMPARISON_SUBJECTS),
        elapsed_s=perf_counter()-started,
        extra={'backend': 'PyTorch/POT CUDA' if use_gpu else 'CPU fallback'},
    )
    set_trial_metrics(trial, aggregates)
    return float(aggregates['mean_intra_inter_ratio'])


## 6. Optimisation et résultats
`n_jobs=1` empêche explicitement plusieurs essais CUDA simultanés.

In [6]:
study = create_reid_study(STUDY_PATH, EXPERIMENT_NAME, seed=SEED)
# Un kernel tué par l'OOM ne clôt pas son essai SQLite. À ce point aucun
# optimize() ne tourne encore dans ce notebook: les RUNNING sont donc orphelins.
n_stale = fail_stale_running_trials(study)
if n_stale: print(f'Nettoyage: {n_stale} essai(s) RUNNING orphelin(s) -> FAIL.')
# Garantit un premier résultat faisable; sans COMPLETE, recréer le sampler avec
# la même seed après chaque crash reproposait toujours les mêmes paramètres.
if not any(t.state == TrialState.COMPLETE for t in study.trials) and not any(t.state == TrialState.WAITING for t in study.trials):
    study.enqueue_trial({'bin_size':8.0,'binning_nb':2,'method':'median','epsilon':0.75})
# Un essai par batch garantit un GC complet et sauvegarde immédiatement.
run_until_complete(
    study, objective, StudyRunPolicy(N_TRIALS, trials_per_batch=1)
)
print(f'Base SQLite : {STUDY_PATH}')


Étude : 120/120 essais COMPLETE; cible restante=0.
Base SQLite : /home/colin/Tractographie/KADMON/notebooks/optuna/studies/optuna_reid.sqlite3


## 7. Résultats bruts de l'étude

> Cet affichage historique contient aussi les valeurs intermédiaires des essais `PRUNED`. Pour le classement valide, utiliser l'analyse filtrée de la section 8.


In [7]:
ratio_best = study.best_trial
print(
    f"Optimum brut du ratio : trial {ratio_best.number} "
    f"(ratio={ratio_best.value:.6f})"
)


Optimum brut du ratio : trial 374 (ratio=0.490418)


## 8. Contour exploratoire initial

Le contour Optuna montre l'effet conjoint des paramètres sur le score.


In [8]:
fig = optuna.visualization.plot_contour(study)
show(fig)


[W 2026-08-26 09:42:17,690] Param binning_nb unique value length is less than 2.
[W 2026-08-26 09:42:17,690] Param binning_nb unique value length is less than 2.
[W 2026-08-26 09:42:17,691] Param binning_nb unique value length is less than 2.
[W 2026-08-26 09:42:17,691] Param binning_nb unique value length is less than 2.
[W 2026-08-26 09:42:17,691] Param binning_nb unique value length is less than 2.
[W 2026-08-26 09:42:17,691] Param binning_nb unique value length is less than 2.


## 9. Analyse des essais observés

Analyse des seuls essais `COMPLETE` déjà présents dans `study` ; le score intra/inter-identité est à minimiser. Les tableaux distinguent l'optimum strict des compromis proches du meilleur score.


In [9]:
complete_trials = [
    trial for trial in study.trials if trial.state == TrialState.COMPLETE
]
rows = [{
    "trial": trial.number,
    "score": trial.value,
    **trial.params,
    "reid_accuracy": trial.user_attrs.get(
        "intra_identity_top1_accuracy", np.nan
    ),
    "mean_n_representatives": trial.user_attrs.get(
        "mean_n_representatives", np.nan
    ),
} for trial in complete_trials]
df = pd.DataFrame(rows).sort_values("score")
best_score = df["score"].min()

columns = [
    "trial", "score", "bin_size", "binning_nb", "method", "epsilon",
    "reid_accuracy", "mean_n_representatives",
]
display(
    df[columns].head(20).style.format({
        "score": "{:.6f}", "bin_size": "{:.0f}",
        "epsilon": "{:.6g}", "reid_accuracy": "{:.0%}",
        "mean_n_representatives": "{:.1f}",
    })
)

reid_best = select_best_reid_trial(study.trials)
display(pd.Series({
    'experiment': EXPERIMENT_NAME,
    'trial': reid_best.number,
    'selection': 'Top-1, rang, ratio, couverture',
    'mean_intra_inter_ratio': reid_best.value,
    **reid_best.params,
    **reid_best.user_attrs,
}, name='meilleur essai RE-ID').to_frame())
reid = reid_best.user_attrs
if "reid_valid_bundles" in reid:
    print("RE-ID validée :", reid["reid_valid_bundles"])
    print("RE-ID échouée :", reid["reid_failed_bundles"])

# À score comparable, epsilon élevé favorise un transport plus régularisé.
tradeoff_rows = []
for threshold in (1, 2, 5):
    candidates = df[df["score"] <= best_score * (1 + threshold / 100)]
    row = candidates.sort_values(
        ["epsilon", "score"], ascending=[False, True]
    ).iloc[0]
    tradeoff_rows.append({
        "seuil (%)": threshold,
        "trial": int(row.trial),
        "score": row.score,
        "écart relatif (%)": 100 * (row.score / best_score - 1),
        "bin_size": row.bin_size,
        "binning_nb": int(row.binning_nb),
        "method": row.method,
        "epsilon": row.epsilon,
        "reid_accuracy": row.reid_accuracy,
        "mean_n_representatives": row.mean_n_representatives,
    })
tradeoff = pd.DataFrame(tradeoff_rows)
display(
    tradeoff.style.format({
        "score": "{:.6f}", "écart relatif (%)": "{:.2f}",
        "bin_size": "{:.0f}", "epsilon": "{:.6g}",
        "reid_accuracy": "{:.0%}", "mean_n_representatives": "{:.1f}",
    })
)

method_summary = df.groupby("method").agg(
    n=("score", "size"),
    meilleur_score=("score", "min"),
    score_médian=("score", "median"),
)
bin_size_summary = df.groupby("bin_size").agg(
    n=("score", "size"),
    meilleur_score=("score", "min"),
    score_médian=("score", "median"),
    représentants_médians=("mean_n_representatives", "median"),
)
display(method_summary.style.format({
    "meilleur_score": "{:.6f}", "score_médian": "{:.6f}",
}))
display(bin_size_summary.style.format({
    "meilleur_score": "{:.6f}", "score_médian": "{:.6f}",
    "représentants_médians": "{:.1f}",
}))


,trial,score,bin_size,binning_nb,method,epsilon,reid_accuracy,mean_n_representatives
99,374,0.490418,8,2,mean,0.0251727,97%,986.4
112,476,0.490435,8,2,mean,0.0251781,97%,986.4
73,224,0.490696,8,2,mean,0.0252617,97%,986.4
32,99,0.490781,8,2,mean,0.0252891,97%,986.4
91,310,0.490863,8,2,mean,0.0253151,97%,986.4
109,447,0.490863,8,2,mean,0.0253151,97%,986.4
96,353,0.490943,8,2,mean,0.0253407,97%,986.4
105,416,0.490954,8,2,mean,0.0253444,97%,986.4
58,161,0.491049,8,2,mean,0.0253745,97%,986.4
111,474,0.491296,8,2,mean,0.0254531,97%,986.4


,meilleur essai RE-ID
experiment,binning_sinkhorn
trial,374
selection,"Top-1, rang, ratio, couverture"
mean_intra_inter_ratio,0.490418
bin_size,8.0
binning_nb,2
method,mean
epsilon,0.025173
backend,PyTorch/POT CUDA
elapsed_s,307.831535


RE-ID validée : ['tractosearch_nn_8_0mm_all_AF_L_m', 'tractosearch_nn_8_0mm_all_AF_R_m', 'tractosearch_nn_8_0mm_all_CC_1_m', 'tractosearch_nn_8_0mm_all_CC_2a_m', 'tractosearch_nn_8_0mm_all_CC_2b_m', 'tractosearch_nn_8_0mm_all_CC_3_m', 'tractosearch_nn_8_0mm_all_CC_4_m', 'tractosearch_nn_8_0mm_all_CC_5_m', 'tractosearch_nn_8_0mm_all_CC_6_m', 'tractosearch_nn_8_0mm_all_CC_7_m', 'tractosearch_nn_8_0mm_all_CG_L_m', 'tractosearch_nn_8_0mm_all_CG_R_m', 'tractosearch_nn_8_0mm_all_CST_R_m', 'tractosearch_nn_8_0mm_all_ICP_L_m', 'tractosearch_nn_8_0mm_all_ICP_R_m', 'tractosearch_nn_8_0mm_all_IFOF_L_m', 'tractosearch_nn_8_0mm_all_IFOF_R_m', 'tractosearch_nn_8_0mm_all_ILF_L_m', 'tractosearch_nn_8_0mm_all_ILF_R_m', 'tractosearch_nn_8_0mm_all_MCP_m', 'tractosearch_nn_8_0mm_all_OR_L_m', 'tractosearch_nn_8_0mm_all_OR_R_m', 'tractosearch_nn_8_0mm_all_SLF_1_L_m', 'tractosearch_nn_8_0mm_all_SLF_1_R_m', 'tractosearch_nn_8_0mm_all_SLF_2_L_m', 'tractosearch_nn_8_0mm_all_SLF_2_R_m', 'tractosearch_nn_8_0mm_al

,seuil (%),trial,score,écart relatif (%),bin_size,binning_nb,method,epsilon,reid_accuracy,mean_n_representatives
0,1,263,0.495007,0.94,8,2,mean,0.0265977,97%,986.4
1,2,330,0.499643,1.88,9,2,mean,0.0314832,97%,707.7
2,5,71,0.512460,4.49,10,2,mean,0.0380141,97%,533.4


,n,meilleur_score,score_médian
method,,,
mean,114,0.490418,0.494942
median,6,0.733139,0.844480


,n,meilleur_score,score_médian,représentants_médians
bin_size,,,,
8.000000,95,0.490418,0.494069,986.4
9.000000,13,0.499643,0.521472,707.7
10.000000,6,0.504176,0.622510,533.4
11.000000,3,0.526496,0.533940,411.5
13.000000,1,0.798516,0.798516,258.9
16.000000,2,0.890443,0.899783,146.3


## 10. Visualisations Optuna

Les importances donnent une vue globale et les coupes montrent l'effet individuel des paramètres. Le contour est limité à `epsilon` et `bin_size` : `binning_nb=2` pour tous les essais terminés, donc ce paramètre ne peut pas être comparé graphiquement.


In [10]:
fig_importance = optuna.visualization.plot_param_importances(study)
show(fig_importance)

fig_contour = optuna.visualization.plot_contour(
    study, params=["epsilon", "bin_size"]
)
show(fig_contour)


## 11. Interprétation des compromis observés

L'étude finale contient **120 essais `COMPLETE`**, ainsi que 448 essais `PRUNED` et 14 essais `FAIL`. Seuls les essais terminés sont utilisés dans les tableaux et l'interprétation; les valeurs intermédiaires des essais élagués ne sont pas des résultats comparables.

Le **trial 374** est l'optimum observé (`score=0,490418`, `bin_size=8 mm`, `binning_nb=2`, `method=mean`, `epsilon≈0,025173`). Son exactitude RE-ID Top-1 est de **96,77 %** (30 bundles sur 31); le seul échec est `CST_L`. Il conserve en moyenne **986,4 représentants** par bundle.

Parmi les solutions proches de l'optimum, augmenter `epsilon` offre davantage de régularisation, tandis qu'augmenter `bin_size` réduit fortement le nombre de représentants et donc le coût des étapes aval :

- à moins de 1 % : trial 263, `score=0,495007`, `bin_size=8 mm`, `epsilon≈0,026598`, 986,4 représentants (+0,94 %);
- à moins de 2 % : trial 330, `score=0,499643`, `bin_size=9 mm`, `epsilon≈0,031483`, 707,7 représentants (+1,88 %);
- à moins de 5 % : trial 71, `score=0,512460`, `bin_size=10 mm`, `epsilon≈0,038014`, 533,4 représentants (+4,49 %).

Les **114 essais `mean`** dominent nettement les 6 essais `median` observés (meilleur score : 0,490418 contre 0,733139). Cette comparaison est toutefois déséquilibrée, car les essais `median` faisables sont rares. De même, tous les essais `COMPLETE` ont `binning_nb=2`; l'étude ne permet donc pas de conclure sur `binning_nb=3`, qui a surtout conduit à des essais élagués ou non faisables avec la contrainte GPU actuelle.

- Le **trial 374 est à la fois le choix RE-ID et le défaut anatomique de KADMON** : il conserve la résolution la plus fine parmi les compromis retenus et utilise la masse Sinkhorn complète.
- Pour un transport légèrement plus régularisé sans changer la compression, considérer le trial 263.
- Pour une future initialisation **LDDMM** avec un compromis précision/coût plus intéressant, privilégier le trial 330 : environ **28 % de représentants en moins** pour seulement +1,88 % sur le score.
- Le trial 71 réduit le nombre de représentants d'environ **46 %**, mais sa perte de score de +4,49 % le réserve aux cas où le coût de calcul domine.
